# Week 1 — Project Definition
**Business context:** A municipal government wants to improve service delivery and resource allocation using its citizen service request records.

**Stakeholder questions:** (1) Which categories create most demand? (2) Which departments resolve requests fastest? (3) How do high-priority requests perform? (4) Which districts generate most requests? (5) What temporal patterns exist? (6) Where are status and closure workflow bottlenecks?

**Dataset:** Municipal_Service_Data.csv (user-supplied; original source city and licensing not specified). This is an analytical case study, not a verified city's official service performance. First-response times, true SLA compliance and causal explanations are not observable.

**Plan:** audit → clean → describe → compare → derive KPIs → diagnose → dashboard → report.

# Week 2 — Dataset Exploration
Inspect shape, data types, uniqueness, missing values, category values and timestamps. Do not assume that a closure timestamp implies a closed status.

In [1]:
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT/'data/raw/Municipal_Service_Data.csv').exists():
    ROOT = ROOT.parent
raw = pd.read_csv(ROOT/'data/raw/Municipal_Service_Data.csv')
print('Shape:', raw.shape)
display(raw.head(), raw.dtypes.to_frame('dtype'), raw.isna().sum().to_frame('missing'))
print('Duplicate IDs:',raw.Service_Request_ID.duplicated().sum())
for c in ['Request_Type','Department','Priority_Level','Status','Ward_District']:
    print('\n',c,raw[c].value_counts(dropna=False).to_dict())

Shape: (2000, 11)


,Service_Request_ID,Request_Type,Description,Department,Priority_Level,Status,Date_Opened,Date_Closed,Ward_District,Latitude,Longitude
0,SR00001,Garbage Collection,Service request related to municipal operations.,Public Works,High,Closed,2022-04-13 19:00:00,2022-04-28 08:00:00,South,49.233907,-123.020945
1,SR00002,Water Leakage,Service request related to municipal operations.,Public Works,Low,Open,2022-05-02 18:00:00,NaN,West,49.259853,-123.019665
2,SR00003,Pothole Repair,Service request related to municipal operations.,Public Works,Low,Closed,2023-10-26 02:00:00,NaN,South,49.216600,-122.918434
3,SR00004,Sewer Issue,Service request related to municipal operations.,Parks & Recreation,Low,Open,2023-01-21 20:00:00,2023-01-28 13:00:00,South,49.157427,-123.070416
4,SR00005,Sewer Issue,Service request related to municipal operations.,Electrical Services,Medium,In Progress,2023-07-15 09:00:00,2023-07-23 05:00:00,Central,49.136821,-122.964447


,dtype
Service_Request_ID,object
Request_Type,object
Description,object
Department,object
Priority_Level,object
Status,object
Date_Opened,object
Date_Closed,object
Ward_District,object
Latitude,float64


,missing
Service_Request_ID,0
Request_Type,0
Description,0
Department,0
Priority_Level,0
Status,0
Date_Opened,0
Date_Closed,185
Ward_District,0
Latitude,0


Duplicate IDs: 0

 Request_Type {'Noise Complaint': 245, 'Road Damage': 232, 'Water Leakage': 228, 'Tree Maintenance': 227, 'Sewer Issue': 221, 'Pothole Repair': 219, 'Street Light Outage': 214, 'Garbage Collection': 208, 'Graffiti Removal': 206}

 Department {'Electrical Services': 420, 'Water Services': 402, 'Parks & Recreation': 401, 'Public Works': 389, 'Sanitation': 388}

 Priority_Level {'Medium': 524, 'Emergency': 502, 'Low': 498, 'High': 476}

 Status {'Closed': 1198, 'In Progress': 419, 'Open': 383}

 Ward_District {'South': 422, 'West': 422, 'North': 394, 'East': 384, 'Central': 378}


# Week 3 — Data Preparation
Parse timestamps with coercion, preserve the original raw file, create valid-resolution and mismatch flags, and never impute unknown closure dates. A valid resolution requires **Status = Closed**, both dates, and a nonnegative elapsed duration. Conflicting records remain in the cleaned data and are flagged for investigation rather than silently deleted.

In [2]:
df = raw.copy()
for c in ['Date_Opened','Date_Closed']:
    df[c]=pd.to_datetime(df[c],errors='coerce')
for c in ['Request_Type','Department','Priority_Level','Status','Ward_District']:
    df[c]=df[c].astype('string').str.strip()
df['Resolution_Days']=(df.Date_Closed-df.Date_Opened).dt.total_seconds()/86400
df['Valid_Resolution']=df.Status.eq('Closed') & df.Resolution_Days.ge(0)
df['Closure_Status_Mismatch']=df.Status.eq('Closed') ^ df.Date_Closed.notna()
df['Open_Month']=df.Date_Opened.dt.to_period('M').astype(str)
df['Open_Weekday']=df.Date_Opened.dt.day_name()
df['Open_Hour']=df.Date_Opened.dt.hour
print('Status/closure mismatch:',df.Closure_Status_Mismatch.sum())
print('Valid resolution records:',df.Valid_Resolution.sum())
(ROOT/'data/cleaned').mkdir(exist_ok=True,parents=True)
df.to_csv(ROOT/'data/cleaned/Municipal_Service_Cleaned.csv',index=False)

Status/closure mismatch: 813
Valid resolution records: 1100
